![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--pii-sanitization-for-production-agents--pii_sanitization_tutorial)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NirDiamant/agents-towards-production/blob/main/tutorials/pii-sanitization-for-production-agents/pii_sanitization_tutorial.ipynb)

# PII Sanitization for Production AI Agents

**The missing layer between user input and your LLM.**

Every production agent processes user text. That text frequently contains sensitive personal information — emails, phone numbers, national IDs, private keys, and financial data. Without a sanitization layer, this PII reaches your LLM provider unfiltered.

This tutorial shows you how to add a pre-LLM PII sanitization hook to any agent pipeline.


## The Problem

A typical agent pipeline today:

```
User: 'My email is john@example.com, call me at +1-555-0123'
         ↓
    Agent processes
         ↓
    LLM Provider ← raw PII arrives here
```

### Why this matters

- **GDPR Article 25**: Privacy by Design requires PII protection before processing
- **EU AI Act (Aug 2026)**: Autonomous agent data governance enforcement begins
- **HIPAA**: Patient data in healthcare agent pipelines must be protected
- **LGPD / CCPA**: Brazil and California extend these requirements globally

The fix is a sanitization hook before your LLM call:

```
User input → [PII Sanitizer] → Agent → LLM Provider
                  ↑
     PII removed before LLM sees it
```


## Three Approaches to PII Sanitization

| Approach | Coverage | Setup | Best for |
|----------|----------|-------|----------|
| Regex patterns | ~70% | Zero | Simple, known formats |
| Local NLP (Presidio) | ~80% | Medium | Air-gapped environments |
| Semantic API (TrustBoost) | ~95% | Minimal | Production agents, multilingual |

We will implement all three and compare results.


In [ ]:
# Install dependencies
!pip install requests langchain langgraph -q

## Approach 1: Regex-based sanitization

Fast, zero dependencies, but misses contextual PII.


In [ ]:
import re

PII_PATTERNS = [
    (r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', 'EMAIL'),
    (r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', 'PHONE'),
    (r'\b\d{3}-\d{2}-\d{4}\b', 'SSN'),
    (r'sk-[a-zA-Z0-9]{32,}', 'API_KEY'),
]

def regex_sanitize(text: str) -> str:
    for pattern, label in PII_PATTERNS:
        text = re.sub(pattern, '[REDACTED]', text)
    return text

# Test
test_input = 'My email is john@example.com and my SSN is 123-45-6789'
print('Input: ', test_input)
print('Output:', regex_sanitize(test_input))

## Approach 2: Semantic API sanitization with TrustBoost

Handles contextual PII, multilingual patterns, and returns structured metadata for audit trails.

**Setup:** No installation required. Use `tx_hash='TRIAL'` for 50 free sanitizations.


In [ ]:
import requests

def trustboost_sanitize(text: str, wallet_id: str = 'my-agent') -> dict:
    """
    Sanitize PII using TrustBoost semantic API.
    Returns sanitized text + safety score + entity list for audit.
    """
    response = requests.post(
        'https://api.trustboost.dev/sanitize',
        json={
            'text': text,
            'tx_hash': 'TRIAL',
            'wallet_address': wallet_id
        },
        timeout=30
    )
    data = response.json()
    return {
        'sanitized': data['data']['sanitized_content'],
        'safety_score': data['data']['safety_score'],
        'risk_category': data['data']['risk_category'],
        'entities': data['data'].get('entities', []),
        'quota_remaining': data['data']['usage_metrics']['quota_remaining']
    }

# Test — English
result = trustboost_sanitize('My email is john@example.com and my SSN is 123-45-6789')
print('Sanitized:    ', result['sanitized'])
print('Safety score: ', result['safety_score'])
print('Risk category:', result['risk_category'])
print('Entities:     ', result['entities'])
print('Quota left:   ', result['quota_remaining'])

In [ ]:
# Test — LATAM Spanish (RFC, CURP, Cedula)
result_es = trustboost_sanitize(
    'Cliente: Juan Lopez, RFC: LOPJ850101ABC, Tel: 55-1234-5678, Email: juan@empresa.com.mx',
    wallet_id='my-agent'
)
print('Input:    Cliente: Juan Lopez, RFC: LOPJ850101ABC, Tel: 55-1234-5678')
print('Output:  ', result_es['sanitized'])
print('Entities:', result_es['entities'])

In [ ]:
# Test — Japanese (My Number)
result_jp = trustboost_sanitize(
    '田中太郎、マイナンバー：123456789012、電話：090-1234-5678',
    wallet_id='my-agent'
)
print('Output:', result_jp['sanitized'])

## Integration with LangChain

Adding sanitization as a preprocessing step before any LLM call.


In [ ]:
from langchain.schema import HumanMessage
from langchain_openai import ChatOpenAI

def create_privacy_aware_agent(llm, wallet_id: str = 'my-agent'):
    """
    Wraps any LangChain LLM with a PII sanitization layer.
    PII is removed before the message reaches the model.
    """
    def invoke_with_sanitization(user_input: str) -> str:
        # Step 1: Sanitize before LLM sees the input
        sanitized = trustboost_sanitize(user_input, wallet_id)
        
        print(f'[PII Guard] Removed {len(sanitized["entities"])} entities'
              f' | Score: {sanitized["safety_score"]}'
              f' | Risk: {sanitized["risk_category"]}')
        
        # Step 2: Send clean input to LLM
        response = llm.invoke([HumanMessage(content=sanitized['sanitized'])])
        return response.content
    
    return invoke_with_sanitization

# Usage example (requires OPENAI_API_KEY)
# llm = ChatOpenAI(model='gpt-4o-mini')
# agent = create_privacy_aware_agent(llm)
# response = agent('Help me email john@company.com about project X')
print('Privacy-aware agent wrapper ready.')

## Integration with LangGraph

Adding sanitization as a dedicated node in your agent graph.


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class AgentState(TypedDict):
    user_input: str
    sanitized_input: str
    safety_score: float
    risk_category: str
    llm_response: str

def sanitize_node(state: AgentState) -> AgentState:
    """Sanitization node — runs before any LLM node."""
    result = trustboost_sanitize(state['user_input'])
    return {
        **state,
        'sanitized_input': result['sanitized'],
        'safety_score': result['safety_score'],
        'risk_category': result['risk_category']
    }

def should_block(state: AgentState) -> str:
    """Route to block if risk is CRITICAL, else continue."""
    if state['risk_category'] == 'CRITICAL':
        return 'block'
    return 'continue'

def block_node(state: AgentState) -> AgentState:
    return {**state, 'llm_response': 'Request blocked: critical PII detected.'}

def llm_node(state: AgentState) -> AgentState:
    # Your LLM call goes here — using sanitized_input, not user_input
    return {**state, 'llm_response': f'Processed: {state["sanitized_input"]}'}

# Build the graph
graph = StateGraph(AgentState)
graph.add_node('sanitize', sanitize_node)
graph.add_node('llm', llm_node)
graph.add_node('block', block_node)
graph.set_entry_point('sanitize')
graph.add_conditional_edges('sanitize', should_block, {'continue': 'llm', 'block': 'block'})
graph.add_edge('llm', END)
graph.add_edge('block', END)
app = graph.compile()

# Test
result = app.invoke({'user_input': 'Send 149 USDC from wallet ABC123 to john@example.com'})
print('Sanitized input:', result['sanitized_input'])
print('Risk category:  ', result['risk_category'])
print('LLM response:   ', result['llm_response'])

## Production Patterns

### Autonomous quota management

TrustBoost returns `quota_remaining` on every response. Use it for autonomous budget management:

```python
result = trustboost_sanitize(text)
if result['quota_remaining'] < 10:
    # Notify operator — quota running low
    notify_operator('TrustBoost quota low — renewal needed')
```

### Audit trail

Every sanitization is logged automatically with safety score, risk category, and entity count — no raw PII stored. Use the returned metadata for your own compliance records:

```python
audit_record = {
    'timestamp': datetime.utcnow().isoformat(),
    'safety_score': result['safety_score'],
    'risk_category': result['risk_category'],
    'entities_removed': len(result['entities']),
    'agent_id': wallet_id
}
```


## Summary

You have implemented a production-ready PII sanitization layer for your agent pipeline:

1. **Regex baseline** — zero dependencies, covers standard formats
2. **Semantic sanitization** — TrustBoost handles contextual and multilingual PII
3. **LangChain integration** — wrapper pattern for any LLM
4. **LangGraph integration** — dedicated sanitization node with routing
5. **Production patterns** — audit trails and quota management

## Resources

- TrustBoost GitHub: https://github.com/teodorofodocrispin-cmyk/TrustBoost-PII-Sanitizer
- Health check: https://api.trustboost.dev/health
- Free preview (no wallet): POST https://api.trustboost.dev/sanitize/preview
- ClawHub: https://clawhub.ai/teodorofodocrispin-cmyk/trustboost-pii-sanitizer
